# Проба пути обучения. Без видеокарты

Этот прогон ничего полезного не считает. Его задача одна, проверить, что
код обучения способен доехать от начала до конца, не упав и не зависнув.

Код обучения до сих пор не исполнялся нигде. Он покрыт тестами, но тесты
проверяют логику, а не то, подружатся ли библиотеки на чужой машине.
Именно на этом стыке проект потерял двое суток.

**Видеокарта не нужна и не просится.** Корпус для оценки урезан до двух
тысяч фрагментов, обучающая выборка до двухсот пар. Квота GPU не тратится
совсем, поэтому прогон может идти рядом с подготовкой.

**Цифры этого прогона ничего не значат.** На двухстах парах и урезанном
корпусе качества нет. Журнал отдельный, чтобы проба не попала в таблицу
рецепта как результат.

Успех это не число, а то, что отработали все шаги. Поднялась модель,
заморозился словарь, собрались наборы, пошло обучение, посчиталась
оценка, записался журнал.

In [ ]:
# Хаб запрещён, и это ставится до первого импорта transformers.
# Иначе библиотека успеет прочитать настройки, и случайное обращение
# к сети не упадёт, а молча встанет.
import os
for ключ in ('HF_HUB_OFFLINE', 'TRANSFORMERS_OFFLINE', 'HF_DATASETS_OFFLINE'):
    os.environ[ключ] = '1'
print('офлайн включён:', {k: os.environ[k] for k in
      ('HF_HUB_OFFLINE', 'TRANSFORMERS_OFFLINE', 'HF_DATASETS_OFFLINE')})

ФРАГМЕНТОВ = 2000   # корпус для оценки
ПАР = 200           # обучающая выборка
МЕТКА = 'a-1ep'     # какой этап проверяем

import time
НАЧАЛО = time.time()


def прошло():
    с = time.time() - НАЧАЛО
    return f'{с // 60:.0f} мин {с % 60:02.0f} с'

## 1. Что есть в окружении

In [ ]:
import torch, transformers, sentence_transformers, datasets, accelerate
print('transformers', transformers.__version__)
print('sentence-transformers', sentence_transformers.__version__)
print('datasets', datasets.__version__, '| accelerate', accelerate.__version__)
print('видеокарта:', torch.cuda.is_available(), '(она тут и не нужна)')

from sentence_transformers import SentenceTransformerTrainer
from sentence_transformers.losses import (CachedMultipleNegativesRankingLoss,
                                          MatryoshkaLoss)
print('всё, что нужно обучению, импортируется')

## 2. Код и данные

In [ ]:
import glob, shutil, zipfile


def find_source(root='/kaggle/input', max_depth=5):
    found, archives = [], []
    for cur, dirs, files in os.walk(root):
        if cur.count('/') - root.count('/') >= max_depth:
            dirs.clear()
        if 'rufin' in dirs:
            stale = ('__notebook__.ipynb' in files or '__output__.json' in files)
            found.append((1 if stale else 0, cur))
        archives += [os.path.join(cur, f) for f in files if f.endswith('.zip')]
    if found:
        found.sort()
        return found[0][1], None
    return (os.path.dirname(archives[0]), archives[0]) if archives else (None, None)


src, archive = find_source()
assert src, 'датасет с кодом не подключён'
print('код:', src)
if archive:
    with zipfile.ZipFile(archive) as z:
        z.extractall('/kaggle/working')
else:
    for имя in os.listdir(src):
        путь, цель = os.path.join(src, имя), os.path.join('/kaggle/working', имя)
        if os.path.isdir(путь):
            shutil.copytree(путь, цель, dirs_exist_ok=True)
        elif имя.endswith(('.py', '.md')):
            shutil.copyfile(путь, цель)

dst = '/kaggle/working/chunks/base.jsonl'
os.makedirs('/kaggle/working/chunks', exist_ok=True)
if os.path.exists(dst):
    os.remove(dst)
найдено = (glob.glob('/kaggle/input/**/chunks_base.jsonl', recursive=True)
           + glob.glob('/kaggle/input/**/chunks_base.jsonl.gz', recursive=True))
assert найдено, 'нет chunks_base.jsonl'
import gzip
opener = gzip.open if найдено[0].endswith('.gz') else open
with opener(найдено[0], 'rb') as fi, open(dst, 'wb') as fo:
    shutil.copyfileobj(fi, fo, length=1 << 20)
print('нарезка:', sum(1 for _ in open(dst, encoding='utf-8')), 'фрагментов')

# Модель ученика обязана быть подключена входом. Качать её нельзя.
модели = glob.glob('/kaggle/input/models/**/config.json', recursive=True)
print('модели входом:', sorted({os.path.dirname(m) for m in модели})[:4])
assert модели, 'модель не подключена: + Add Input -> Models'
print('прошло', прошло())

## 3. Проба

Нулевая точка и один этап обучения. Журнал и папка весов отдельные.

In [ ]:
import subprocess

cmd = ['python', 'run_phase_d.py', '--step', 'all',
       '--tags', МЕТКА,
       '--limit-chunks', str(ФРАГМЕНТОВ),
       '--limit-train', str(ПАР),
       '--batch-size', '32',
       '--weights', '/kaggle/working/proba/weights',
       '--journal', '/kaggle/working/proba/journal.json']
print('запуск:', ' '.join(cmd), flush=True)
код = subprocess.run(cmd, cwd='/kaggle/working').returncode
print('\nкод возврата:', код, '| прошло', прошло())

## 4. Что вышло

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working')
from rufin.training.journal import Journal

путь = '/kaggle/working/proba/journal.json'
if os.path.exists(путь):
    j = Journal.load(путь)
    print(j.table())
    print('\nэтапов в журнале:', len(j.entries))
    print('ЦИФРЫ НИЧЕГО НЕ ЗНАЧАТ: корпус урезан, выборка урезана')
else:
    print('журнала нет, путь не прошёл')

веса = '/kaggle/working/proba/weights'
if os.path.isdir(веса):
    for папка in sorted(os.listdir(веса)):
        размер = sum(os.path.getsize(os.path.join(б, и))
                     for б, _, ф in os.walk(os.path.join(веса, папка)) for и in ф)
        print(f'веса {папка}: {размер / 1048576:.0f} МБ')

shutil.rmtree('/kaggle/working/chunks', ignore_errors=True)
print('\nвсего', прошло())